Übung: talosctl
-----------

- - -

In der Übung erstellen wir einen Kubernetes Cluster in einem Container. 

Dazu benötigen wir zuerst eine Konfigurationsdatei welche die Anzahl control und Worker Nodes festlegt. Hier ein control und drei Worker Nodes

- - -

Zuerst müssen wir taloctl installieren


In [ ]:
%%bash
wget https://github.com/siderolabs/talos/releases/download/v1.8.2/talosctl-linux-amd64 -O talosctl
chmod +x talosctl
sudo mv talosctl /usr/local/bin/

Cluster innerhalb Docker erstellen

In [ ]:
! talosctl cluster create

Und schauen wie die Distribution zusammengesetzt ist

In [ ]:
%%bash
talosctl cluster show
kubectl get pods -A

- - -

### Vergleich mit der microk8s Kubernetes Distribution

Zum Vergleich die Kubernetes Dienste von der [microk8s](https://microk8s.io/) Kubernetes Distribution.

Weil auf dem control schon zuviele Prozesse laufen, schauen wir uns einen der Worker an:

In [ ]:
! sudo microk8s kubectl get pods -n kube-system
! echo "---------------------------------------------------------------------------------------------"
! systemctl list-units | grep microk8s | grep -v mounted

Wir stellen fest, dass sich die Kubernetes Distributionen hauptsächlich 
* in der Art der verwendeteten Produkte (z.B. für Netzwerk kindnet vs. calico, Key-Value Server etcd vs. dqlite)
* ob die System Prozesse als Container oder Linux System Dienste laufen
  
unterscheiden.

- - -

### Aufräumen (erst nach Portweiterleitung ausführen) 

Dazu löschen wir den `talosctl` Kubernetes Cluster und wechseln auf unseren `microk8s` Kubernetes cluster

In [ ]:
! talosctl cluster destroy
! kubectl config use-context microk8s

- - -
### Portweiterleitung (Port Forwarding) - nur für Fortgeschrittene!


Ein wichtiger Aspekt bei der Arbeit mit Kubernetes ist die Fähigkeit, auf die im Cluster ausgeführten Dienste von aussen zuzugreifen. 

Dies wird häufig durch die Verwendung von NodePorts erreicht, die es ermöglichen, einen bestimmten Port auf den Knoten im Cluster einem Port im Container zuzuordnen. 

Allerdings stellt sich die Herausforderung, diese NodePorts vom lokalen Rechner aus zugänglich zu machen, insbesondere wenn der Cluster in einem Docker-Container läuft, wie es bei kind der Fall ist.

Wenn wir also z.B. die Microservices aus dem aus dem [REST-Beispiel](demo/Microservices-REST.ipynb) starten, werden wir nicht auf die Applikation bzw. den Port zugreifen können.

Mit einer einfach Portweiterleitung (Port Forwarding) zur VM umgehen wir diesen Umstand:

In [ ]:
%%bash
kubectl config use-context kind-docker-kind
echo http://"$(cat ~/data/server-ip)":8888/webshop
kubectl port-forward --namespace ms-rest --address 0.0.0.0 service/webshop 8888:8080

Mittels des Stop Buttons (oben) können wir die Weiterleitung stoppen.

Weiter geht es oben mit **Aufräumen** und Anschliessend kehren wir zum `microk8s` Cluster zurück.